# Test Battery — Algorithm Visualization

Debug and visualize the behavior of the online optimization algorithms using
mock instances built from `tests/online/builders.py`.

### What you get
- **Gantt chart** — driver schedules (preassigned vs. new labors vs. driver moves)
- **KPI table** — algorithm metrics
- **Route map** — geographic representation of the assignments

### How to use
Edit the **Configuration** cell, then *Run All*.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import folium

_PROJECT_ROOT = Path("..").resolve()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from src.optimization.algorithms.registry import get_algorithm
from src.optimization.settings.model_params import ModelParams

from src.analysis.solution_evaluation import _wkt_to_latlon, build_route_map

from tests.online.builders import (
    DIST_METHOD,
    MAX_ITERATIONS,
    _MODEL_PARAMS,
    make_master_data,
    build_offline_simple,
    build_offline_two_drivers,
    build_insert_empty_schedule,
    build_insert_at_end,
    build_insert_in_gap,
    build_buffer_no_freeze,
    build_buffer_full_freeze,
    build_buffer_partial_freeze,
)

print("Imports OK")

Imports OK


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
#
# ALGORITHM: "OFFLINE" | "INSERT" | "BUFFER_REACT"
# SCENARIO:  see scenario map below for valid names per algorithm
#
ALGORITHM = "BUFFER_REACT"  # "OFFLINE" | "INSERT" | "BUFFER_REACT"
SCENARIO  = "no_freeze"  # see SCENARIO_MAP in the next cell

In [14]:
# ── Scenario map & algorithm runner ────────────────────────────────────────────

# Maps ALGORITHM + SCENARIO → builder call → run → (results_df, metrics, artifacts, extra)
# extra = {\"preassigned_labors_df\": ..., \"new_labors_df\": ...} for INSERT/BUFFER

_COMMON = dict(
    distance_method=DIST_METHOD,
    time_method="speed_based",
    n_processes=None,
    precompute_distances=False,
    log_progress=False,
    model_params=_MODEL_PARAMS,
)

preassigned_labors_df = pd.DataFrame()
preassigned_moves_df  = pd.DataFrame()   # always defined; populated for INSERT/BUFFER
new_labors_df = pd.DataFrame()

if ALGORITHM == "OFFLINE":
    _builders = {
        "simple":      build_offline_simple,
        "two_drivers": build_offline_two_drivers,
    }
    assert SCENARIO in _builders, f"SCENARIO must be one of {list(_builders)}"
    labors_df, directorio_df = _builders[SCENARIO]()
    new_labors_df = labors_df
    params = {
        **_COMMON,
        "max_iterations_by_city": MAX_ITERATIONS,
        "master_data": make_master_data(directorio_df),
    }
    results_df, metrics, artifacts = get_algorithm("OFFLINE", params).solve(labors_df)

elif ALGORITHM == "INSERT":
    _builders = {
        "empty":  build_insert_empty_schedule,
        "at_end": build_insert_at_end,
        "in_gap": build_insert_in_gap,
    }
    assert SCENARIO in _builders, f"SCENARIO must be one of {list(_builders)}"
    new_labors_df, preassigned_labors_df, preassigned_moves_df, directorio_df = _builders[SCENARIO]()
    params = {
        **_COMMON,
        "max_iterations": MAX_ITERATIONS,
        "master_data": make_master_data(directorio_df),
        "preassigned": {"labors_df": preassigned_labors_df, "moves_df": preassigned_moves_df},
    }
    results_df, metrics, artifacts = get_algorithm("INSERT", params).solve(new_labors_df)

elif ALGORITHM == "BUFFER_REACT":
    _builders = {
        "no_freeze":      (build_buffer_no_freeze,      0),
        "full_freeze":    (build_buffer_full_freeze,    300),
        "partial_freeze": (build_buffer_partial_freeze, 180),
    }
    assert SCENARIO in _builders, f"SCENARIO must be one of {list(_builders)}"
    builder_fn, time_previous_freeze = _builders[SCENARIO]
    new_labors_df, preassigned_labors_df, preassigned_moves_df, directorio_df, decision_time = builder_fn()
    params = {
        **_COMMON,
        "max_iterations_by_city": MAX_ITERATIONS,
        "time_previous_freeze": time_previous_freeze,
        "decision_time": decision_time,
        "master_data": make_master_data(directorio_df),
        "preassigned": {"labors_df": preassigned_labors_df, "moves_df": preassigned_moves_df},
    }
    results_df, metrics, artifacts = get_algorithm("BUFFER_REACT", params).solve(new_labors_df)
else:
    raise ValueError(f"Unknown ALGORITHM={ALGORITHM!r}")

moves_df = artifacts.get("moves_df", pd.DataFrame())

print(f"✓ {ALGORITHM}/{SCENARIO}: {len(results_df)} result rows, {len(moves_df)} move rows")
print(f"  Metrics: {metrics}")

✓ BUFFER_REACT/no_freeze: 3 result rows, 9 move rows
  Metrics: {'algorithm': 'BUFFER_REACT', 'elapsed_seconds': 0.027553208001336316, 'cities_processed': 1, 'postponed_labors_count': 0, 'moves_rows': 9, 'frozen_labors_count': 0, 'reassigned_labors_count': 2, 'new_labors_count': 1, 'time_previous_freeze': 0}


In [15]:
# ── Raw output preview ─────────────────────────────────────────────────────────
print("=== results_df ===")
display(results_df[[c for c in [
    "labor_id", "service_id", "labor_sequence", "assigned_driver",
    "schedule_date", "actual_start", "actual_end",
    "map_start_point", "map_end_point",
] if c in results_df.columns]])

if not preassigned_labors_df.empty:
    print("\n=== preassigned_labors_df ===")
    display(preassigned_labors_df[[c for c in [
        "labor_id", "service_id", "assigned_driver", "schedule_date", "actual_start", "actual_end",
    ] if c in preassigned_labors_df.columns]])

=== results_df ===


,labor_id,service_id,labor_sequence,assigned_driver,schedule_date,actual_start,actual_end,map_start_point,map_end_point
0,90001,9001,1,9001,2026-03-01 08:00:00-05:00,2026-03-01 08:00:00-05:00,2026-03-01 09:22:31.522133-05:00,POINT (-74.0 4.6),POINT (-74.0 4.779986)
1,90002,9002,1,9001,2026-03-01 10:10:00-05:00,2026-03-01 11:07:35.754081-05:00,2026-03-01 12:30:07.276214-05:00,POINT (-74.0 5.09496),POINT (-74.0 5.274946)
2,90003,9003,1,9001,2026-03-01 12:00:00-05:00,2026-03-01 13:00:08.493920-05:00,2026-03-01 14:13:17.129265-05:00,POINT (-74.0 5.364939),POINT (-74.0 5.499928)



=== preassigned_labors_df ===


,labor_id,service_id,assigned_driver,schedule_date,actual_start,actual_end
0,90001,9001,9001,2026-03-01 08:00:00-05:00,2026-03-01 08:00:00-05:00,2026-03-01 09:07:31.522133-05:00
1,90002,9002,9001,2026-03-01 10:10:00-05:00,2026-03-01 10:10:00-05:00,2026-03-01 11:17:31.522133-05:00


---
## Gantt Charts — Three-Stage View

Three charts show the scheduling progression:

1. **Stage 1 — Preassigned Schedule**: Frozen/preassigned labors with their original driver moves
2. **Stage 2 — After Insertion**: All labors placed, with original preassigned moves as context
3. **Stage 3 — Final Schedule**: All labors with the complete final move timeline

Color coding:
- **Blue** — preassigned / frozen labors
- **Orange** — newly inserted / optimized labors
- **Amber** — driver moves (inter-service travel)
- **Light grey** — free time

In [16]:
# ── Three-stage Gantt helper ────────────────────────────────────────────────────

_COLOR_PREASSIGNED = "rgba(55, 115, 200, 0.85)"    # blue
_COLOR_NEW         = "rgba(245, 130, 32,  0.88)"    # orange
_COLOR_DRIVER_MOVE = "rgba(250, 200, 50,  0.85)"    # amber
_COLOR_FREE_TIME   = "rgba(190, 190, 190, 0.40)"    # light grey

_BASE_TS = pd.Timestamp("2026-03-01", tz="America/Bogota")

def _make_gantt_fig(title: str, labors_df: pd.DataFrame, moves_df: pd.DataFrame,
                    preassigned_ids: set) -> go.Figure:
    """Build a single Gantt figure for the given labors and (optional) moves."""
    fig = go.Figure()

    # ── Move / free-time bars ──────────────────────────────────────────────────
    # Only render FREE_TIME and DRIVER_MOVE rows; skip the _labor rows that
    # build_driver_movements() emits (they would overlap with labor bars).
    if not moves_df.empty and "actual_start" in moves_df.columns:
        for _, row in moves_df.iterrows():
            cat = str(row.get("labor_category", "")).upper()
            if cat not in ("FREE_TIME", "DRIVER_MOVE"):
                continue
            driver = str(row.get("assigned_driver", "?"))
            t_start = pd.Timestamp(row["actual_start"])
            t_end   = pd.Timestamp(row["actual_end"])
            if pd.isna(t_start) or pd.isna(t_end) or t_start >= t_end:
                continue
            color = _COLOR_DRIVER_MOVE if cat == "DRIVER_MOVE" else _COLOR_FREE_TIME
            label = "Driver move" if cat == "DRIVER_MOVE" else "Free time"
            fig.add_trace(go.Bar(
                name=label,
                x=[(t_end - t_start).total_seconds() / 60],
                y=[driver],
                base=[(t_start - _BASE_TS).total_seconds() / 60],
                orientation="h",
                marker_color=color,
                showlegend=False,
                hovertemplate=(
                    f"<b>{label}</b><br>"
                    f"Driver: {driver}<br>"
                    f"Start: {t_start.strftime('%H:%M')}<br>"
                    f"End: {t_end.strftime('%H:%M')}"
                    "<extra></extra>"
                ),
            ))

    # ── Labor bars ────────────────────────────────────────────────────────────
    if not labors_df.empty and "actual_start" in labors_df.columns:
        for _, row in labors_df.iterrows():
            driver = str(row.get("assigned_driver", "unassigned"))
            t_start = pd.Timestamp(row["actual_start"])
            t_end   = pd.Timestamp(row["actual_end"])
            if pd.isna(t_start) or pd.isna(t_end) or t_start >= t_end:
                continue
            labor_id_str = str(row.get("labor_id", ""))
            is_preassigned = labor_id_str in preassigned_ids
            color = _COLOR_PREASSIGNED if is_preassigned else _COLOR_NEW
            bar_label = "Preassigned" if is_preassigned else "New labor"
            fig.add_trace(go.Bar(
                name=bar_label,
                x=[(t_end - t_start).total_seconds() / 60],
                y=[driver],
                base=[(t_start - _BASE_TS).total_seconds() / 60],
                orientation="h",
                marker_color=color,
                showlegend=False,
                hovertemplate=(
                    f"<b>{bar_label}</b><br>"
                    f"Labor: {labor_id_str}<br>"
                    f"Service: {row.get('service_id', '?')}<br>"
                    f"Driver: {driver}<br>"
                    f"Schedule: {pd.Timestamp(row.get('schedule_date', t_start)).strftime('%H:%M')}<br>"
                    f"Start: {t_start.strftime('%H:%M')}<br>"
                    f"End: {t_end.strftime('%H:%M')}"
                    "<extra></extra>"
                ),
            ))

    # ── Legend entries ─────────────────────────────────────────────────────────
    for lbl, clr in [
        ("Preassigned / frozen", _COLOR_PREASSIGNED),
        ("New labor",            _COLOR_NEW),
        ("Driver move",          _COLOR_DRIVER_MOVE),
        ("Free time",            _COLOR_FREE_TIME),
    ]:
        fig.add_trace(go.Bar(
            name=lbl, x=[None], y=[None],
            marker_color=clr, showlegend=True, orientation="h",
        ))

    # ── Determine y-axis drivers ───────────────────────────────────────────────
    _all_drivers = set()
    if not labors_df.empty and "assigned_driver" in labors_df.columns:
        _all_drivers |= set(labors_df["assigned_driver"].dropna().astype(str))
    if not moves_df.empty and "assigned_driver" in moves_df.columns:
        _all_drivers |= set(moves_df["assigned_driver"].dropna().astype(str))
    _n_drivers = max(len(_all_drivers), 1)

    fig.update_layout(
        title=f"{ALGORITHM} / {SCENARIO} — {title}",
        barmode="overlay",
        xaxis=dict(
            title="Time (Bogotá)",
            tickvals=[(i * 60) for i in range(24)],
            ticktext=[f"{i:02d}:00" for i in range(24)],
            range=[7 * 60, 20 * 60],
        ),
        yaxis=dict(title="Driver"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        height=max(300, 80 * (_n_drivers + 2)),
    )
    return fig


# ── Preassigned IDs set ────────────────────────────────────────────────────────
_preassigned_ids = set()
if not preassigned_labors_df.empty and "labor_id" in preassigned_labors_df.columns:
    _preassigned_ids = set(preassigned_labors_df["labor_id"].dropna().astype(str))

# ── Stage 2 moves: final moves filtered to preassigned labor_ids ───────────────
# Using preassigned_moves_df here would reference the ORIGINAL positions of
# reassigned labors (e.g. L2 at 10:10 in partial_freeze), causing amber bars to
# misalign with the orange labor bars in results_df (which use the new 10:52
# position). Instead, filter moves_df so we show the correct final-state moves
# for only the preassigned labors, without the new labor's moves.
_moves_stage2 = (
    moves_df[moves_df["labor_id"].astype(str).isin(_preassigned_ids)]
    if not moves_df.empty and _preassigned_ids and "labor_id" in moves_df.columns
    else pd.DataFrame()
)

# ── Stage 3 moves: supplement moves_df with preassigned moves that precede it ──
# Algorithms (especially BUFFER_REACT) may omit preassigned moves for the period
# before the first unfrozen labor. Prepend any preassigned moves whose window
# ends at or before the earliest timestamp already in moves_df.
if not preassigned_moves_df.empty and not moves_df.empty and "actual_start" in moves_df.columns:
    _cutoff = moves_df["actual_start"].dropna().min()
    _pre_only = preassigned_moves_df[
        preassigned_moves_df["actual_end"].dropna() <= _cutoff
    ]
    _moves_stage3 = (
        pd.concat([_pre_only, moves_df], ignore_index=True)
        if not _pre_only.empty else moves_df
    )
else:
    _moves_stage3 = moves_df

# ── Stage 1: Preassigned Schedule (with original moves) ───────────────────────
if preassigned_labors_df.empty:
    print("Stage 1 — Preassigned Schedule: no preassigned labors (OFFLINE scenario)")
else:
    _make_gantt_fig(
        "Stage 1 — Preassigned Schedule",
        preassigned_labors_df,
        preassigned_moves_df,
        _preassigned_ids,
    ).show()

# ── Stage 2: After Insertion (all labors + final moves for preassigned only) ───
_make_gantt_fig(
    "Stage 2 — After Insertion (preassigned moves as context)",
    results_df,
    _moves_stage2,
    _preassigned_ids,
).show()

# ── Stage 3: Final Schedule (complete move timeline) ──────────────────────────
_make_gantt_fig(
    "Stage 3 — Final Schedule",
    results_df,
    _moves_stage3,
    _preassigned_ids,
).show()

---
## KPI Metrics

In [6]:
# ── KPI table ──────────────────────────────────────────────────────────────────
kpi_df = pd.DataFrame(
    [{"metric": k, "value": v} for k, v in metrics.items()]
).set_index("metric")
display(kpi_df)

,value
metric,
algorithm,BUFFER_REACT
elapsed_seconds,0.020333
cities_processed,1
postponed_labors_count,0
moves_rows,9
frozen_labors_count,1
reassigned_labors_count,1
new_labors_count,1
time_previous_freeze,180


---
## Route Maps — Preassigned vs. Final

Two side-by-side maps showing the route before and after optimization.

- **Left — Preassigned**: the frozen schedule before new labors are added
- **Right — Final**: the complete schedule after optimization

- **Numbered circles** — labor start points (color-coded per labor)
- **Open rings** — intermediate labor end points
- **F marker** — final labor end point
- **Green home icon** — driver home position
- **Thick lines** — service legs (OSRM routing when available, dashed fallback)
- **Thin lines** — driver move legs
- Use the **dropdown** to switch drivers when multiple are present.

In [18]:
# ── Side-by-side route maps: Preassigned vs. Final ────────────────────────────
import ipywidgets as widgets
from IPython.display import display as _display

_MAP_W, _MAP_H = 640, 520

# ── points_lookup for each view ───────────────────────────────────────────────
def _build_points_lookup(df: pd.DataFrame) -> dict:
    return {
        str(row["labor_id"]): (row.get("map_start_point"), row.get("map_end_point"))
        for _, row in df.iterrows()
    } if not df.empty else {}

_points_lookup_pre   = _build_points_lookup(preassigned_labors_df)
_points_lookup_final = _build_points_lookup(results_df)

# ── rows lists (build_route_map expects driver_id key) ────────────────────────
# Also normalise labor_type: builders use numeric code "12", but build_route_map
# checks against _VT_LABOR_TYPES = {"alfred_initial_transport", "alfred_transport"}.
# Without this, is_vt=False for every row and all legs get the light move colour
# instead of the dark service colour.
def _to_map_rows(df: pd.DataFrame) -> list:
    rows = []
    for d in df.to_dict("records"):
        d["driver_id"] = str(d.get("assigned_driver", ""))
        d["labor_id"]  = str(d.get("labor_id", ""))
        if d.get("labor_category") == "VEHICLE_TRANSPORTATION":
            d["labor_type"] = "alfred_transport"
        rows.append(d)
    return rows

_map_rows_pre   = _to_map_rows(preassigned_labors_df)
_map_rows_final = _to_map_rows(results_df)

# ── driver home WKT lookup ────────────────────────────────────────────────────
_driver_home_wkt: dict = {}
if not directorio_df.empty:
    for _, _drv in directorio_df.iterrows():
        try:
            _lat = float(_drv["latitud"])
            _lon = float(_drv["longitud"])
            _driver_home_wkt[str(_drv["driver_id"])] = f"POINT ({_lon} {_lat})"
        except Exception:
            pass

# ── all drivers (union of both views) ─────────────────────────────────────────
_drv_set = set()
for _df in (preassigned_labors_df, results_df):
    if not _df.empty and "assigned_driver" in _df.columns:
        _drv_set |= set(_df["assigned_driver"].dropna().astype(str))
_all_drivers = sorted(_drv_set)

if not _all_drivers:
    print("No geographic data to display.")
else:
    # ── Widget layout (mirrors compare_solutions.ipynb) ────────────────────────
    _drv_dropdown = widgets.Dropdown(
        options=_all_drivers,
        description="Driver:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="300px"),
    )

    _out_pre   = widgets.Output(layout=widgets.Layout(width=f"{_MAP_W}px"))
    _out_final = widgets.Output(layout=widgets.Layout(width=f"{_MAP_W}px"))

    _map_row = widgets.HBox([
        widgets.VBox([
            widgets.HTML("<h4 style='text-align:center;margin:4px 0'>Preassigned</h4>"),
            _out_pre,
        ]),
        widgets.VBox([
            widgets.HTML("<h4 style='text-align:center;margin:4px 0'>Final</h4>"),
            _out_final,
        ]),
    ])

    def _render_maps(driver_id):
        home_wkt = _driver_home_wkt.get(str(driver_id))

        _out_pre.clear_output(wait=True)
        with _out_pre:
            if not _map_rows_pre:
                print("No preassigned route for this scenario.")
            else:
                try:
                    _display(build_route_map(
                        services=None,
                        rows=_map_rows_pre,
                        driver_id=driver_id,
                        driver_home_wkt=home_wkt,
                        points_lookup=_points_lookup_pre,
                        label="Preassigned",
                        reverse_palette=False,
                        map_width=_MAP_W,
                        map_height=_MAP_H,
                    ))
                except Exception as _e:
                    print(f"No preassigned route for driver {driver_id}: {_e}")

        _out_final.clear_output(wait=True)
        with _out_final:
            try:
                _display(build_route_map(
                    services=None,
                    rows=_map_rows_final,
                    driver_id=driver_id,
                    driver_home_wkt=home_wkt,
                    points_lookup=_points_lookup_final,
                    label="Final",
                    reverse_palette=True,
                    map_width=_MAP_W,
                    map_height=_MAP_H,
                ))
            except Exception as _e:
                print(f"Map error for driver {driver_id}: {_e}")

    _drv_dropdown.observe(lambda change: _render_maps(change["new"]), names="value")
    _render_maps(_drv_dropdown.value)
    _display(_drv_dropdown, _map_row)

Dropdown(description='Driver:', layout=Layout(width='300px'), options=('9001',), style=DescriptionStyle(descri…